# ***vLLM Setup (Local Model Serving)***

If you would like to run inference locally, you can leverage the **vLLM** library.

First, install vLLM on your machine:

```bash
pip install vllm
```
Next, download your desired model from Hugging Face (or another model repository).

Once the model is downloaded, you can serve it locally using the command line. When serving your model with vLLM, make sure to specify the following:

- **Model path**: The local path to your downloaded model  
- **Host and port**: These define where the model will be served and will be used as the `base_url` when creating a client with the OpenAI library  
- **Maximum model length**: The maximum number of tokens supported during inference  
- **Quantization setting**: Choose an option that is compatible with your available GPU resources  

### Example

```bash
python -m vllm.entrypoints.openai.api_server 
    --model=/workspace/models/llama70B 
    --host=127.0.0.1 
    --port=8000 
    --max_model_len=10000
    --quantization=bitsandbytes
```

Once the server is running, you can connect to it using the OpenAI-compatible API interface.

# ***Dependencies***

## ***Installation***

In [ ]:
!pip install faiss-gpu
!pip install -U sentence-transformers
!pip install openai
!pip install -q pandas
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q seaborn
!pip install openpyxl
!pip install sentencepiece
!pip install protobuf

In [ ]:
!pip install --upgrade typing_extensions pydantic pydantic-core openai 
# then restart kernel

## ***Imports***

In [ ]:
from huggingface_hub import login

login(token="Your HuggingFace Token")

In [ ]:

import pandas as pd
from sklearn.metrics import accuracy_score , classification_report, confusion_matrix , f1_score, accuracy_score,precision_recall_fscore_support , precision_score , recall_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import time
# import faiss
from sentence_transformers import SentenceTransformer
import warnings
# from prompt_manager import PromptConstructor
from huggingface_hub import login
from  tqdm import tqdm_notebook, tqdm
import csv
import os
from openai import OpenAI
import openai

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
tqdm.pandas()

# ***Configuration and Global Variables***

## ***Keys***

In [ ]:
openai.api_key =  'Your OpenAI key'
# If using Openrouter, set the following variables
openrouter_base_url = 'https://openrouter.ai/api/v1'
openrouter_key = 'Your openRouter Key'

huggingface_key = ''

vllm_api_key = "EMPTY"
vllm_base_url = "http://127.0.0.1:8000/v1"

## ***Variables and Paths***

In [ ]:
CLASS_To_LABEL = {1 : "AD" , 0 : "Healthy"}

root_dir = 'dir/'
data_dir = os.path.join(root_dir, 'data')
train_orig_data_path = os.path.join(data_dir, 'train.csv')
valid_orig_data_path = os.path.join(data_dir, 'validation.csv')
test_orig_data_path = os.path.join(data_dir, 'Test_DePiC.xlsx')

train_embed_data_path = os.path.join(data_dir, 'train_with_embed.pkl')
valid_embed_data_path = os.path.join(data_dir, 'validation_with_embed.pkl')
test_embed_data_path = os.path.join(data_dir, 'test_with_embed.pkl')

results_dir = '/results'

## ***Prompts***

In [ ]:
instruct_prompts = {}

instruct_prompts["prompt_v1"]= (
    'You are an expert in cognitive health and language analysis. You will analyze a spoken language transcript from a person describing the "cookie theft" picture. This is not written text but a transcription of spontaneous speech.'
    '\nAnalyze the provided transcript and classify it into one of two categories: "Healthy" for a healthy cognitive state or "AD" for Alzheimer disease.'
    '\nProvide only the label ("Healthy" or "AD") as the output. Do not include explanations or additional text.'
    'The output should be in JSON format, like {"label": "predicted label"}'
)

# ***Helper Functions***

## ***Data Loading and Evaluation Utilities***

In [ ]:
def load_dataframe(file_path):
    _, ext = os.path.splitext(file_path)
    ext = ext.lower()

    if ext == '.csv':
        return pd.read_csv(file_path)
    elif ext in ['.xls', '.xlsx']:
        return pd.read_excel(file_path)
    elif ext == '.pkl':
        return pd.read_pickle(file_path)
    else:
        raise ValueError(f"Unsupported file extension: {ext}")

def save_dataframe(dataframe, dataframe_save_path):
    dataframe.to_pickle(dataframe_save_path)
    print(f'Saved dataframe to {dataframe_save_path}.')

def evaluate_model_metrics(y_true, y_pred):
    """
    Compute and print classification metrics (accuracy, precision, recall, F1) for classes 0 and 1,
    including macro averages, while handling an optional -1 error class.

    Args:
        y_true (array-like): Ground truth class labels.
        y_pred (array-like): Predicted class labels.
    
    Returns:
        dict: Dictionary of computed metrics.
    """
    
    # Calculate accuracy over all classes, including -1
    accuracy = accuracy_score(y_true, y_pred)

    # Calculate metrics for class 0
    precision_0 = precision_score(y_true, y_pred, labels=[0], average='macro', zero_division=0)
    recall_0 = recall_score(y_true, y_pred, labels=[0], average='macro', zero_division=0)
    f1_0 = f1_score(y_true, y_pred, labels=[0], average='macro', zero_division=0)
    # f1_bin = f1_score(y_true, y_pred, zero_division=0)

    # Calculate metrics for class 1
    precision_1 = precision_score(y_true, y_pred, labels=[1], average='macro', zero_division=0)
    recall_1 = recall_score(y_true, y_pred, labels=[1], average='macro', zero_division=0)
    f1_1 = f1_score(y_true, y_pred, labels=[1], average='macro', zero_division=0)

    # Calculate macro average for classes 0 and 1 (excluding class -1)
    precision_macro = precision_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0)

    # Print the results
    print(f"Accuracy (including -1 class): {accuracy:.4f}")
    # print(f"f1 (binary): {f1_bin:.4f}")

    # Metrics for class 0
    print(f"Class healthy - Precision: {precision_0:.4f}, Recall: {recall_0:.4f}, F1 Score: {f1_0:.4f}")

    # Metrics for class 1
    print(f"Class ADRD - Precision: {precision_1:.4f}, Recall: {recall_1:.4f}, F1 Score: {f1_1:.4f}")

    # Macro average (excluding class -1)
    print(f"Macro Precision : {precision_macro:.4f}")
    print(f"Macro Recall : {recall_macro:.4f}")
    print(f"Macro F1 Score : {f1_macro:.4f}")

    return {
        "accuracy": accuracy,
        "precision_class_0": precision_0,
        "recall_class_0": recall_0,
        "f1_class_0": f1_0,
        "precision_class_1": precision_1,
        "recall_class_1": recall_1,
        "f1_class_1": f1_1,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro
    }


def save_results(results , model_name , test_file, sub_dir='llama3B'):
    """
    Save model evaluation results to a CSV file.

    Args:
        results (dict): Dictionary of evaluation metrics (keys as column names, values as row data).
        model_name (str): Name of the evaluated model.
        test_file (str): Identifier for the test dataset/file.
        sub_dir (str, optional): Subdirectory for storing results. Default is 'llama3B'.
    """

    # Specify the CSV file path
    csv_file = f'{results_dir}/{reason_dir}/{sub_dir}/results_{model_name}_{test_file}.csv'

    # Check if the file exists to determine whether to write the header
    file_exists = os.path.isfile(csv_file)

    # Open the file in append mode
    with open(csv_file, mode='a', newline='') as file:
        writer = csv.writer(file)

        # Write the header if the file doesn't exist
        if not file_exists:
            writer.writerow(results.keys())

        # Write the results row
        writer.writerow(results.values())
    app_name = results["approach_name"]
    print(f"Results for approach '{app_name}' have been saved to {csv_file}.")


def plot_metrics(metrics , maetric_name , title):
    """
    Plot metric values against the number of few-shot demonstrations.

    Args:
        metrics (list or array): Metric values for each few-shot setting.
        maetric_name (str): Name of the metric being plotted.
        title (str): Plot title suffix (dataset or experiment description).
    """
    
    few_shots = np.array(list(range(1, len(metrics) + 1)))*2
    plt.plot(few_shots, metrics, marker='o', linestyle='-')
    plt.title(f"{maetric_name} over demonestrations - {title}")
    plt.xlabel('number of demonestrations')
    plt.ylabel(maetric_name)
    plt.grid(True)
    plt.show()


# ***Component 1: ICL Framework***

## ***Demonstration Selection***

This section implements strategies for selecting relevant few-shot demonstrations based on embedding similarity.

In [ ]:
def get_k_similar_example(train_df , test_df , n = 1  , distance = "nearest" ):
    """
    Retrieve k training examples most similar, furthest, average-based, or random relative to test embeddings.

    Args:
        train_df (pd.DataFrame): Training data containing 'embedding' column (list/array of vectors).
        test_df (pd.DataFrame): Test data containing 'embedding' column (list/array of vectors).
        n (int, optional): Rank of the neighbor to retrieve (1 = closest/furthest). Default is 1.
        distance (str, optional): Similarity mode:
            - "nearest": k-th most similar training example(s) to each test example.
            - "furthest": k-th least similar training example(s) to each test example.
            - "average": Training example closest to the mean training embedding.
            - "random": Similarity search using random embeddings for training data.

    Returns:
        pd.DataFrame: Matching training examples with a 'similarity' column.
    """

    train_embeddings = np.array(train_df['embedding'].tolist())
    test_embeddings = np.array(test_df['embedding'].tolist())


    # Compute similarity
    cos_sim_matrix = cosine_similarity(test_embeddings, train_embeddings)

    if distance == "nearest":
        # Find the indices that would sort each row
        sorted_indices = np.argsort(cos_sim_matrix, axis=1)

        # Retrieve indices of k nearest neighbors
        k_nearest_neighbors_indices = sorted_indices[:, -1 * n]

        # Retrieve the second k neighbors for each test
        k_nearest_neighbors = train_df.iloc[k_nearest_neighbors_indices]

        k_nearest_similarity_values = cos_sim_matrix[np.arange(len(cos_sim_matrix)), k_nearest_neighbors_indices]
        k_nearest_neighbors['similarity'] = k_nearest_similarity_values

        return k_nearest_neighbors

    elif  distance == "furthest":
        sorted_indices = np.argsort(cos_sim_matrix, axis=1)
        # Retrieve indices of k furthest neighbors
        k_furthest_neighbors_indices = sorted_indices[:, n-1]

        # Retrieve the second furthest neighbors for each test
        k_furthest_neighbors = train_df.iloc[k_furthest_neighbors_indices]

        k_furthest_similarity_values = cos_sim_matrix[np.arange(len(cos_sim_matrix)), k_furthest_neighbors_indices]
        k_furthest_neighbors["similarity"] = k_furthest_similarity_values

        return k_furthest_neighbors

    elif distance == "average":
        mean_train_embedding = np.mean(train_embeddings, axis=0).reshape(1, -1)

        # Compute cosine similarities between the mean embedding and all training embeddings
        cosine_similarities = cosine_similarity(mean_train_embedding, train_embeddings)

        # Sort the indices of cosine similarities in descending order (highest similarity first)
        sorted_indices = np.argsort(cosine_similarities[0])[::-1]

        # The index of the k nearest neighbor (the first is the nearest)
        k_closest_vector_index = sorted_indices[n-1]

        # Retrieve the k nearest neighbor
        k_nearest_neighbor = train_df.iloc[k_closest_vector_index]

        k_nearest_neighbor = pd.concat([k_nearest_neighbor] * len(test_embeddings), axis=1).T

        # Reset the index if needed
        k_nearest_neighbor.reset_index(drop=True, inplace=True)

        # Compute similarity
        cos_sim_matrix = cosine_similarity(test_embeddings, train_embeddings[k_closest_vector_index].reshape(1, -1))

        # Add a new column with the similarity value
        k_nearest_neighbor['similarity'] = cos_sim_matrix.reshape( -1)

        return k_nearest_neighbor

    elif distance == "random":

        copy_train = train_df.copy()
         # Set the seed for reproducibility
        seed = 12
        np.random.seed(seed)

        # Convert the embeddings to a numpy array
        train_embeddings = np.array(copy_train['embedding'].tolist())

        # Get the shape of the embeddings
        embedding_shape = train_embeddings.shape

        # Generate random vectors with the same shape and data type as the original embeddings
        random_vectors = np.random.randn(embedding_shape[0], embedding_shape[1])

        # Convert the random vectors back to a list of lists, similar to train_embeddings
        random_vectors_list = random_vectors.tolist()
        copy_train["embedding"] = random_vectors_list

        return get_k_similar_example(copy_train , test_df , n = 1  , distance = "nearest" )

In [ ]:
def get_demonestrations(number_of_demonestrations , train_data , test_data , distance = "nearest"):
    """
    Generate a set of demonstration examples for each test instance by retrieving 
    similar training examples from both class 0 and class 1.

    Args:
        number_of_demonestrations (int): Number of nearest/furthest/other examples per class to retrieve.
        train_data (pd.DataFrame): Training data with 'label' (0 or 1) and 'embedding' columns.
        test_data (pd.DataFrame): Test data with 'embedding' column.
        distance (str, optional): Similarity selection mode passed to get_k_similar_example 
                                   ('nearest', 'furthest', 'average', 'random'). Default is "nearest".
    
    Returns:
        list of pd.DataFrame: For each test sample, a DataFrame of retrieved demonstrations 
                              sorted by similarity.
    """
    
    instances_zeros = []
    instances_ones = []
    demonestrations = []
    train_data_zero = train_data[train_data["label"] == 0]
    train_data_ones = train_data[train_data["label"] == 1]

    for j in range(1,number_of_demonestrations+1):
        instances_zero = get_k_similar_example(train_data_zero , test_data ,n = j , distance = distance)
        instances_one = get_k_similar_example(train_data_ones , test_data ,n = j , distance = distance)
        instances_zeros.append(instances_zero)
        instances_ones.append(instances_one)

    for index_test in range(len(test_data)):
        if number_of_demonestrations == 0:
            demonestrations_together = []
        else:
            demonestrations_zero = [instance.iloc[index_test] for instance in instances_zeros]
            demonestrations_ones = [instance.iloc[index_test] for instance in instances_ones]
            demonestrations_zero = pd.DataFrame(demonestrations_zero)
            demonestrations_ones = pd.DataFrame(demonestrations_ones)
            demonestrations_together = pd.concat([demonestrations_zero , demonestrations_ones])
            demonestrations_together = demonestrations_together.sort_values(by='similarity', ascending=True)
        demonestrations.append(demonestrations_together)

    return demonestrations

## ***Embedding Generation***

In [ ]:
# Load data
train_data = pd.read_csv(train_orig_data_path)
valid_data = pd.read_csv(valid_orig_data_path)
test_data = pd.read_excel(test_orig_data_path)

# Load Model and encode transcriptions
model = SentenceTransformer("BAAI/bge-large-en-v1.5" , device="cuda" )
embeddings_train = model.encode(train_data["text"].to_list() , show_progress_bar=True , device="cuda")
embeddings_test = model.encode(test_data["text"].to_list() , show_progress_bar=True , device="cuda")
embeddings_valid = model.encode(valid_data["text"].to_list() , show_progress_bar=True , device="cuda")

train_data["embedding"] = embeddings_train.tolist()
test_data["embedding"] = embeddings_test.tolist()
valid_data["embedding"] = embeddings_valid.tolist()

model = model.to('cpu')
torch.cuda.empty_cache()

# Save embeddings
save_dataframe(train_data, train_embed_data_path)
save_dataframe(test_data, test_embed_data_path)
save_dataframe(valid_data, valid_embed_data_path)

## ***Prompt Engineering***

### ***Prompt Constructor***

In [ ]:
class PromptConstructor:
    """
    Utility class for constructing prompts for chat-based language models using their 
    tokenizer's chat template, with optional caching of tokenizers.

    Methods:
        get_tokenizer(model_name): Retrieve (and cache) the tokenizer for the given model.
        get_start_of_assistant(tokenizer, chat_messages): Extract the assistant's starting token 
            sequence from a chat template.
        apply_chat_template(messages, model_name, output_force=None): Apply the model's chat 
            template to messages, optionally forcing a specific assistant output start.
    """
    
    tokenizer_cache = {}

    def __init__(self):
        pass

    @classmethod
    def get_tokenizer(cls, model_name):
        if model_name not in cls.tokenizer_cache:
            cls.tokenizer_cache[model_name] = AutoTokenizer.from_pretrained(model_name)
        return cls.tokenizer_cache[model_name]

    @staticmethod
    def get_start_of_assistant(tokenizer, chat_messages):
        """
        Get the starting token sequence for the assistant's generation.
        """

        def apply_chat_template(template):
            return tokenizer.apply_chat_template(template, tokenize=False)

        def extract_assistant_tokens(full_template, partial_template):
            # Compare full vs. partial templates to isolate the assistant's starting tokens
            full_result = apply_chat_template(full_template)
            partial_result = apply_chat_template(partial_template)

            if partial_result not in full_result:
                # Often triggered when the tokenizer handles system role differently
                raise ValueError(
                    "There is some problem with tokenizer , it may from supporting system role. \n It is better to not use system role and check again to occure error or not")

            # Remove the partial portion and stop at '!!!' placeholder
            return full_result.replace(partial_result, "").split("!!!")[0]

        if chat_messages[0].get("role") == "system":
            # Full template includes a system prompt before user and assistant
            full_template = [
                {"role": "system", "content": "---"},
                {"role": "user", "content": "###"},
                {"role": "assistant", "content": "!!!"}
            ]
            partial_template = [
                {"role": "system", "content": "---"},
                {"role": "user", "content": "###"}
            ]
            return extract_assistant_tokens(full_template, partial_template)

        elif chat_messages[0].get("role") == "user":
            # No system role — assistant follows the user directly
            full_template = [
                {"role": "user", "content": "###"},
                {"role": "assistant", "content": "!!!"}
            ]
            partial_template = [
                {"role": "user", "content": "###"}
            ]
            return extract_assistant_tokens(full_template, partial_template)

        else:
            raise ValueError(f"Unrecognized role: {chat_messages[0].get('role')}")

    def apply_chat_template(self,
                            messages: str,
                            model_name: str,
                            output_force: str = None):
        tokenizer = self.get_tokenizer(model_name)

        if tokenizer.chat_template:
            chat_template = tokenizer.apply_chat_template(messages, tokenize=False)
            if output_force is not None:
                # Append forced starting sequence after assistant's initial tokens
                output_starter = PromptConstructor.get_start_of_assistant(tokenizer, messages)
                chat_template += output_starter
                chat_template += output_force
        else:
            # Fallback for models without a chat template
            chat_template = f"""

### Instruction:
{messages}

### {output_force}"""

        return chat_template


### ***Inference Prompt Generation***

In [ ]:
def make_prompt(instruct_prompt  , test_sample , demonestrations = [] ,  column_reason = "reason" ,  has_reason = False , has_sys_role = True):
    """
    Construct a chat-formatted prompt for classification, optionally including few-shot demonstrations.

    Args:
        instruct_prompt (str): Instruction text for the model.
        test_sample (pd.Series): Test example containing 'text' and 'label' (label used in demonstrations only).
        demonestrations (pd.DataFrame, optional): Few-shot examples with 'text', 'label', and optionally a reason column.
        column_reason (str, optional): Column name in demonstrations containing reasoning text. Default is "reason".
        has_reason (bool, optional): Whether to include reasoning text from demonstrations. Default is False.
        has_sys_role (bool, optional): Whether to format the prompt with a system role. Default is True.

    Returns:
        list of dict: Chat messages formatted for the model's chat API.
    """
    
    add_dem_prompt = "\n\nHere are some example cases for your guidance: \n\n" if len(demonestrations) else ""
    if len(demonestrations) > 0:
            for _,dem in demonestrations.iterrows():
                res = f'"reason" : "{dem[column_reason]}" ,' if has_reason else ""
                dem = f'\nExample:\n\nParticipant transcript: "{dem["text"]}" , output: {{{res} "label": "{CLASS_To_LABEL[dem["label"]]}" }}\n\n'
                add_dem_prompt += dem
    test_template = "\n\nNow classify following example:\n\n" + f'\nExample:\n\nParticipant transcript: "{test_sample["text"]}" '
    if has_sys_role:
        return [{"role" : "system" , "content" : instruct_prompt + add_dem_prompt } ,
                {"role" : "user" , "content" : test_template}]
    else:
        return [{"role" : "user" , "content" : instruct_prompt + add_dem_prompt + test_template }]


## ***Few-Shot Inference and Post-Processing***

In [ ]:
def few_shots(train_data, test_data, test_file="validation", sub_dir='llama3B'):
    """
    Run few-shot evaluation for a model across varying numbers of demonstrations,
    recording metrics, logs, and plots.

    Args:
        train_data (pd.DataFrame): Training data with embeddings, labels, and optional reasoning.
        test_data (pd.DataFrame): Test data with embeddings and labels.
        test_file (str, optional): Identifier for the test dataset when saving results/logs. Default is "validation".
        sub_dir (str, optional): Subdirectory for storing results and logs. Default is "llama3B".
    """
    
    f1scores = []
    accs = []
    f1_class_1 = []
    for i in range(1, max_number_of_demonestraions + 1):
        f1scores_itt = []
        accs_itt = []
        f1_class_1_itt = []
        for itt in range(1, 2):
            all_info = []
            model_answers = []
            model_time_completion = []

            instruct_prompt = instruct_prompts[prompt_version]
            # Retrieve a set of demonstrations for each test example based on similarity distance
            demonestraions = get_demonestrations(i, train_data, test_data, distance=distance_strategy)

            for index_test, (idx, row_test) in enumerate(tqdm_notebook(test_data.iterrows(), total=len(test_data))):
                # Construct the prompt for the current test case, inserting the chosen demonstrations
                prompt_messages = make_prompt(
                    instruct_prompt,
                    row_test,
                    demonestraions[index_test],
                    column_reason=column_reason,
                    has_reason=reason,
                    has_sys_role=has_sys_role
                )

                if chat_compeletion:
                    # Force the model output to begin with a specific JSON structure depending on reasoning flag
                    output_force = 'output: {{"reason" : ' if reason else 'output: {{"label" : '
                    prompt = prompt_constructor.apply_chat_template(
                        messages=prompt_messages,
                        model_name=hf_model_name,
                        output_force=output_force
                    )
                    start_time = time.time()
                    model_generate_response = LLM_response(prompt)
                else:
                    start_time = time.time()
                    model_generate_response = LLM_response(prompt_messages)

                # Track execution time for performance evaluation
                end_time = time.time()
                execution_time = end_time - start_time
                model_generate_response["time"] = execution_time

                outputs = model_generate_response["content"]
                model_answers.append(outputs)
                all_info.append(model_generate_response)
                time.sleep(5)  # Avoids hitting API rate limits

            time_consumptions = [x["time"] for x in all_info]

            # Convert raw model outputs into final answer format (e.g., extracting label or reason)
            answers = post_process_answers(model_answers, reason)
            print(f"{model_name} {i}-shot {distance_strategy} :")

            model_only_name = model_name.split("/")[-1].strip() + f'_{itt}'
            calculated_metrics = evaluate_model_metrics(answers, test_data["label"].to_list())

            # Include config details in results naming for reproducibility
            calculated_metrics["approach_name"] = (
                f"{model_only_name}_{prompt_version}_{0}_shots_"
                f"{'with_reson_' + column_reason if reason else 'No_reason'}"
            )
            save_results(calculated_metrics, model_only_name, test_file, sub_dir)

            f1scores_itt.append(calculated_metrics["macro_f1"])
            accs_itt.append(calculated_metrics["accuracy"])
            f1_class_1_itt.append(calculated_metrics["f1_class_1"])

            print(f"mean time consumption is: {np.mean(time_consumptions):.4f}")

            # Store model predictions alongside test data for later analysis
            model_only_name = model_name.split("/")[-1].strip() + f'_{itt}'
            test_data[
                f"{model_only_name}_{prompt_version}_{i*2}_shots_{distance_strategy}_"
                f"{'with_reson_' + column_reason if reason else 'No_reason'}"
            ] = answers

            # Save detailed log with model responses and timings
            pd.DataFrame(all_info).to_csv(
                f"{results_dir}/{reason_dir}/{sub_dir}/log_info_"
                f"{test_file}_{model_only_name}_{prompt_version}_{i*2}_shots_{distance_strategy}_"
                f"{'with_reson_' + column_reason if reason else 'No_reason'}.csv"
            )

        f1scores.append(np.mean(f1scores_itt))
        accs.append(np.mean(accs_itt))
        f1_class_1.append(np.mean(f1_class_1_itt))

    # Plot performance metrics across different numbers of shots
    plot_metrics(f1_class_1, "F1-score class ADRD", f"{i*2}shot-{model_only_name}-{distance_strategy}")
    plot_metrics(accs, "accuracy", f"{i*2}shot-{model_only_name}-{distance_strategy}_")
    plot_metrics(f1scores, "f1 score macro", f"{i*2}shot-{model_only_name}-{distance_strategy}")

    print("F1 score ADRD:")
    print(f1_class_1)
    print("Accuracy of model:")
    print(accs)
    print("F1 score macro")
    print(f1scores)


In [ ]:
def LLM_response(prompt):
    """
    Calls the OpenAI chat completion API and returns the model response
    along with token usage information.

    Args:
        prompt (list): List of message dictionaries formatted for chat API.

    Returns:
        dict: {
            "content": str,                 # Model generated text
            "prompt_tokens": int,           # Tokens used in prompt
            "completeion_tokens": int,      # Tokens used in completion
            "total_number_tokens": int      # Total tokens used
        }
    """
    res = client.chat.completions.create(
        model=OPENAI_MODEL_NAME,
        messages=prompt,
        temperature=0,          # deterministic output
        max_tokens=8000,        # upper bound on response length
        stream=False
    )

    content = res.choices[0].message.content  # extract generated message text

    # token usage statistics from API response
    prompt_token = res.usage.prompt_tokens
    completeion_tokens = res.usage.completion_tokens
    total_number_tokens = res.usage.total_tokens

    return {
        "content": content,
        "prompt_tokens": prompt_token,
        "completeion_tokens": completeion_tokens,
        "total_number_tokens": total_number_tokens
    }


def post_process_answers(answers, reason=False):
    """
    Converts raw model text predictions into numeric class labels.

    Args:
        answers (list of str): Model-generated text predictions.
        reason (bool): Unused flag (kept for compatibility).

    Returns:
        list of int: Predicted labels (0, 1, or -1 if unmatched).
    """
    label_answers = []

    for predicted in tqdm_notebook(answers):  # progress bar iteration
        predicted_lower = predicted.lower()

        # Match class 0 using label text or fallback keyword ('he')
        if CLASS_To_LABEL[0].lower().strip() in predicted_lower or 'he' in predicted_lower:
            label_answers.append(0)

        # Match class 1 using full label or partial split before apostrophe
        elif (CLASS_To_LABEL[1].lower().strip() in predicted_lower or
              CLASS_To_LABEL[1].split("'")[0].lower().strip() in predicted_lower):
            label_answers.append(1)

        else:
            print("We find an error : ", predicted)  # unmatched prediction
            label_answers.append(-1)

    return label_answers

# ***Experimental Evaluation Across Models***

## ***Load Data***

In [ ]:
train_data = load_dataframe(train_embed_data_path)
valid_data = load_dataframe(valid_embed_data_path)
test_data = load_dataframe(test_embed_data_path)

train_data = train_data[['id', 'text', 'label', 'embedding']]
valid_data = valid_data[['id', 'text', 'label', 'embedding']]
test_data = test_data[['id', 'text', 'label', 'embedding']]

## ***GPT***

In [ ]:
OPENAI_API_KEY = openai.api_key # "EMPTY"
OPENAI_BASE_URL = "http://127.0.0.1:8000/v1"
OPENAI_MODEL_NAME = "gpt-4o-2024-08-06"

column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = model_name =  "gpt-4o-2024-08-06"
OPENAI_MODEL_NAME = model_name
hf_model_name = model_name =  "gpt-4o-2024-08-06"

reason_dir = 'no_reson_valid'
sub_dir = 'gpt4o'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=OPENAI_API_KEY)#, base_url=OPENAI_BASE_URL)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "nearest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

## ***DeepSeek***

In [ ]:
column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = "deepseek/deepseek-r1"
OPENAI_MODEL_NAME = model_name
hf_model_name = "deepseek/deepseek-r1"

reason_dir = 'no_reson_valid'
sub_dir = 'deepseekR1'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=openrouter_key, base_url=openrouter_base_url)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "nearest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

## ***Gemini 2.0 Flash***

In [ ]:
column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = "google/gemini-2.0-flash-001"
OPENAI_MODEL_NAME = model_name
hf_model_name = "google/gemini-2.0-flash-001"

reason_dir = 'no_reson_valid'
sub_dir = 'gemini2flash'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=openrouter_key, base_url=openrouter_base_url)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "nearest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

## ***Llama 405***

In [ ]:
column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = "meta-llama/llama-3.1-405b-instruct"
OPENAI_MODEL_NAME = model_name
hf_model_name = "meta-llama/llama-3.1-405b-instruct"

reason_dir = 'no_reson_valid'
sub_dir = 'gemini2flash'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=openrouter_key, base_url=openrouter_base_url)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "nearest"
column_reason = "reason_mdl"
reason = False
prompt_version = "prompt_v1"
few_shots()

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
column_reason = "reason_mdl"
reason = False
prompt_version = "prompt_v1"
few_shots()

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
column_reason = "reason_mdl"
reason = False
prompt_version = "prompt_v1"
few_shots()

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
column_reason = "reason_mdl"
reason = False
prompt_version = "prompt_v1"
few_shots()

## ***Llama 70B***

In [ ]:
column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = 'meta-llama/llama-3.3-70b-instruct'
OPENAI_MODEL_NAME = model_name
hf_model_name = 'unsloth/Llama-3.3-70B-Instruct'

reason_dir = 'no_reson_valid'
sub_dir = 'llama70B'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=openrouter_key, base_url=openrouter_base_url)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "nearest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

## ***MedAlpaca 7B***

In [ ]:
column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = 'medalpaca/medalpaca-7b'
OPENAI_MODEL_NAME = model_name
hf_model_name = 'medalpaca/medalpaca-7b'

reason_dir = 'no_reson_valid'
sub_dir = 'medalpaca'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=openrouter_key, base_url=openrouter_base_url)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "nearest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

## ***Llama 8B***

In [ ]:
column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = 'meta-llama/llama-3.1-8b-instruct'
OPENAI_MODEL_NAME = model_name
hf_model_name = 'unsloth/Meta-Llama-3.1-8B-Instruct'

reason_dir = 'no_reson_valid'
sub_dir = 'llama8B'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=openrouter_key, base_url=openrouter_base_url)

In [ ]:
max_number_of_demonestraions = 0
distance_strategy = "nearest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

## ***Ministral8B***

In [ ]:
column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = 'mistralai/ministral-8b'
OPENAI_MODEL_NAME = model_name
hf_model_name = 'mistralai/Ministral-8B-Instruct-2410'

reason_dir = 'no_reson_valid'
sub_dir = 'ministral'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=openrouter_key, base_url=openrouter_base_url)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "nearest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

## ***Llama 3B***

In [ ]:
column_reason = "reason_mdl"
reason = False
chat_compeletion = False
has_sys_role = False

model_name = 'meta-llama/llama-3.2-3b-instruct'
OPENAI_MODEL_NAME = model_name
hf_model_name = 'unsloth/Llama-3.2-3B-Instruct'

reason_dir = 'no_reson_valid'
sub_dir = 'llama3B'
os.makedirs(os.path.join(results_dir, reason_dir, sub_dir), exist_ok=True)
client = OpenAI(api_key=openrouter_key, base_url=openrouter_base_url)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "nearest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "furthest"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "average"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)

In [ ]:
max_number_of_demonestraions = 6
distance_strategy = "random"
prompt_version = "prompt_v1"
few_shots(train_data=train_data, test_data=valid_data, sub_dir=sub_dir)